# Phase 3 — Contrôle de l'acquisition

## Objectif

Ce workbook constitue le contrôle maître de la Phase 3. Il valide progressivement la gouvernance, les acquisitions RAW, les préparations nécessaires à l’acquisition, la traçabilité et la couverture des sources avant de clôturer la phase.

Principes de validation :

- **Registry-driven** : la configuration est lue depuis le Source Registry ;
- **RAW-first** : les réponses fournisseurs sont conservées avant préparation métier ;
- **lineage explicite** : les dépendances entre datasets sont contrôlées ;
- **pas de sélection arbitraire** : les ambiguïtés sont conservées et documentées ;
- **périmètre du registre** : le contrôle porte sur les jeux inscrits. Il n'ouvre pas une source absente du registre ;
- **validation avant clôture** : la Phase 3 n’est considérée comme terminée qu’après stabilisation de toutes les sources nécessaires.

> Les cours de marché viennent de Yahoo Finance, `DS-003`. Un titre sans historique complet reste acquis ; il est écarté seulement à la construction du portefeuille.


# 1. Imports et chemins projet

> Les chemins doivent être résolus depuis la racine du projet, sans dépendre du répertoire depuis lequel le notebook est lancé.

# 2. Gouvernance du registre des sources

Le Source Registry comporte deux couches complémentaires :

- **Sources** : gouvernance des fournisseurs et sources externes ;
- **Datasets** : contrats techniques de production, dépendances, stockage et activation.

In [1]:
import io
import json
import re
import sqlite3
import sys
import zipfile
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display

PROJECT_ROOT = Path.cwd().resolve()
for candidate in [PROJECT_ROOT, *PROJECT_ROOT.parents]:
    if (candidate / "src" / "config.py").exists():
        PROJECT_ROOT = candidate
        break
else:
    raise FileNotFoundError(
        "Project root not found from the notebook directory."
    )

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import (
    DATABASE_FILE,
    SOURCE_REGISTRY_FILE,
    TECHNICAL_LOG,
)

if not SOURCE_REGISTRY_FILE.exists():
    raise FileNotFoundError(
        SOURCE_REGISTRY_FILE
    )

sources_registry = pd.read_excel(
    SOURCE_REGISTRY_FILE,
    sheet_name="Sources",
)

datasets_registry = pd.read_excel(
    SOURCE_REGISTRY_FILE,
    sheet_name="Datasets",
)

sources_count = len(
    sources_registry
)

datasets_count = len(
    datasets_registry
)

print(
    "Sources:",
    sources_registry.shape,
)

print(
    "Datasets:",
    datasets_registry.shape,
)

if (
    sources_count > 0
    and datasets_count > 0
):
    status = "PASS"
    message = (
        f"Le Source Registry contient **{sources_count} sources** "
        f"et **{datasets_count} datasets configurés**. "
        "Les deux feuilles de gouvernance sont non vides."
    )
else:
    status = "FAIL"
    message = (
        f"Le Source Registry contient **{sources_count} sources** "
        f"et **{datasets_count} datasets**. "
        "Au moins une feuille de gouvernance est vide."
    )

display(
    Markdown(
        f"""

{message}

**Résultat : {status}**
"""
    )
)


Sources: (4, 7)
Datasets: (7, 15)




Le Source Registry contient **4 sources** et **7 datasets configurés**. Les deux feuilles de gouvernance sont non vides.

**Résultat : PASS**


In [2]:
def get_dataset_path(dataset_id: str) -> Path:
    """
    Resolve a dataset output path from the Dataset Registry.
    """

    matches = datasets_registry.loc[
        datasets_registry["Dataset ID"].eq(
            dataset_id
        )
    ]

    if len(matches) != 1:
        raise ValueError(
            f"Expected exactly one Registry entry "
            f"for {dataset_id}; found {len(matches)}."
        )

    dataset = matches.iloc[0]

    storage_directory = str(
        dataset["Storage Directory"]
    ).strip()

    output_file_name = str(
        dataset["Output File Name"]
    ).strip()

    if not storage_directory:
        raise ValueError(
            f"{dataset_id}: Storage Directory is required."
        )

    if not output_file_name:
        raise ValueError(
            f"{dataset_id}: Output File Name is required."
        )

    return (
        PROJECT_ROOT
        / storage_directory
        / output_file_name
    )

## 2.1 Sources actives

In [3]:
source_columns = [
    col for col in [
        "Source ID", "Provider", "Reliability", "Backup Source", "Status", "Comments"
    ] if col in sources_registry.columns
]

sources_registry[source_columns]

,Source ID,Provider,Reliability,Backup Source,Status,Comments
0,SRC-001,iShares / BlackRock,High,NaN,Active,Benchmark officiel du projet utilisé comme réf...
1,SRC-002,Yahoo Finance,Medium,NaN,Active,Source principale utilisée pour l'acquisition ...
2,SRC-003,Swiss National Bank (SNB),High,NaN,Active,Source officielle retenue pour la conversion d...
3,SRC-004,"Annual Reports, Sustainability Reports & Corpo...",Medium,Public Sustainability Reports,Active,Source générique utilisée pour documenter les ...


## 2.2 Datasets configurés

In [4]:
dataset_columns = [
    "Dataset ID", "Source ID", "Dataset Name", "Production Method", "Connector",
    "Access Endpoint", "Parameters", "Input Dataset IDs", "Output File Name",
    "Storage Directory", "Frequency", "Load Enabled", "Quality Enabled",
    "Chronology Enabled", "Active"
]

missing_dataset_columns = sorted(set(dataset_columns) - set(datasets_registry.columns))
print("Missing Dataset columns:", missing_dataset_columns)

datasets_registry[[c for c in dataset_columns if c in datasets_registry.columns]]

Missing Dataset columns: []


,Dataset ID,Source ID,Dataset Name,Production Method,Connector,Access Endpoint,Parameters,Input Dataset IDs,Output File Name,Storage Directory,Frequency,Load Enabled,Quality Enabled,Chronology Enabled,Active
0,DS-000,SRC-001,Benchmark Holdings,Download,NaN,https://www.blackrock.com/varnish-api/blk-one0...,{},[],benchmark_holdings.xls,data/01_raw/benchmark,Periodic,False,False,False,True
1,DS-001,SRC-001,Security Candidates,Preparation,NaN,NaN,"{\n ""worksheet"": ""Holdings"",\n ""asset_class_...","[""DS-000""]",security_candidates.csv,data/02_processed/securities,Periodic,True,False,False,True
2,DS-002,SRC-001,Security Master,Preparation,NaN,NaN,{},"[""DS-001""]",securities_master.csv,data/02_processed/securities,On demand,True,True,False,True
3,DS-003,SRC-002,Market Prices,Package,yfinance,NaN,"{\n ""symbol_column"": ""yahoo_symbol"",\n ""name...","[""DS-002""]",market_prices.csv,data/01_raw/market,Daily,True,True,True,True
4,DS-004,SRC-003,FX Rates,API,snb,https://data.snb.ch/api/cube,"{\n ""cube_id"": ""devkum"",\n ""language"": ""en"",...","[""DS-002""]",fx_rates.csv,data/01_raw/fx,Monthly,True,True,True,True
5,DS-005,SRC-003,SARON,API,snb,https://data.snb.ch/api/cube,"{\n ""cube_id"": ""zimoma"",\n ""language"": ""en"",...",[],saron.csv,data/01_raw/macro,Monthly,True,True,True,True
6,DS-006,SRC-004,ODD Assessment,Preparation,NaN,NaN,{},"[""DS-002""]",odd_assessment.csv,data/01_raw/odd,Annual,True,False,False,True


## 2.3 Intégrité Source ID

In [5]:
unknown_source_ids = (
    set(
        datasets_registry[
            "Source ID"
        ]
        .dropna()
        .astype(str)
    )
    - set(
        sources_registry[
            "Source ID"
        ]
        .dropna()
        .astype(str)
    )
)

if not unknown_source_ids:
    status = "PASS"
    message = (
        "Tous les `Source ID` utilisés dans la feuille "
        "Datasets existent dans la feuille Sources."
    )
else:
    status = "FAIL"
    message = (
        f"**{len(unknown_source_ids)} Source ID inconnu(s)** "
        "détecté(s) : "
        f"`{sorted(unknown_source_ids)}`."
    )

display(
    Markdown(
        f"""
{message}

**Résultat : {status}**
"""
    )
)


Tous les `Source ID` utilisés dans la feuille Datasets existent dans la feuille Sources.

**Résultat : PASS**


## 2.4 Production Methods et dépendances

In [6]:
print("Production Methods:")
print(datasets_registry["Production Method"].value_counts(dropna=False))
print()
print("Active datasets:")
active_mask = datasets_registry["Active"].astype(str).str.upper().isin(["TRUE", "1"])
datasets_registry.loc[active_mask, ["Dataset ID", "Dataset Name", "Production Method", "Input Dataset IDs"]]


Production Methods:
Production Method
Preparation    3
API            2
Download       1
Package        1
Name: count, dtype: int64

Active datasets:


,Dataset ID,Dataset Name,Production Method,Input Dataset IDs
0,DS-000,Benchmark Holdings,Download,[]
1,DS-001,Security Candidates,Preparation,"[""DS-000""]"
2,DS-002,Security Master,Preparation,"[""DS-001""]"
3,DS-003,Market Prices,Package,"[""DS-002""]"
4,DS-004,FX Rates,API,"[""DS-002""]"
5,DS-005,SARON,API,[]
6,DS-006,ODD Assessment,Preparation,"[""DS-002""]"


# 3. Gouvernance SQLite

La base SQLite conserve le registry technique et la traçabilité des exécutions. Les tables métier évoluent avec les datasets chargés et ne doivent pas être supposées à l’avance.

In [7]:
if DATABASE_FILE.exists():
    with sqlite3.connect(DATABASE_FILE) as connection:
        technical_tables = pd.read_sql(
            """
            SELECT name
            FROM sqlite_master
            WHERE type='table'
            ORDER BY name
            """,
            connection,
        )

    display(technical_tables)

else:
    print(
        "Database not found yet:",
        DATABASE_FILE,
    )

,name
0,acquisition_log
1,dataset_registry
2,fx_rates
3,market_prices
4,odd_assessment
5,quality_control_result
6,quality_metric_result
7,quality_run
8,saron
9,schema_migrations


# 4. DS-000 / SRC-001 — Feuille de positions
On vérifie que le benchmark officiel a été acquis dans la couche RAW et on identifie automatiquement sa structure réelle pour la préparation DS-001.

In [8]:
benchmark_file = get_dataset_path(
    "DS-000"
)

if not benchmark_file.exists():
    raise FileNotFoundError(
        benchmark_file
    )

print(
    "Benchmark file:",
    benchmark_file,
)

Benchmark file: C:\Users\MameBARBOSAMONTEIRO\projet01\data\01_raw\benchmark\benchmark_holdings.xls


In [9]:
def inspect_file_structure(path: Path) -> dict:
    """
    Detect the physical file format from its content and inspect
    worksheet names when the detected format supports worksheets.
    """

    if not isinstance(path, Path):
        raise TypeError(
            "path must be a pathlib.Path."
        )

    if not path.exists():
        raise FileNotFoundError(path)

    if not path.is_file():
        raise ValueError(
            f"Path is not a file: {path}"
        )

    with path.open("rb") as handle:
        content = handle.read()

    result = {
        "format": None,
        "extension": path.suffix,
        "worksheets": [],
    }

    # Office Open XML / XLSX
    if content.startswith(b"PK\x03\x04"):
        result["format"] = "Office Open XML / XLSX"

        try:
            with zipfile.ZipFile(
                io.BytesIO(content)
            ) as archive:
                workbook_xml = archive.read(
                    "xl/workbook.xml"
                ).decode(
                    "utf-8",
                    errors="ignore",
                )

            result["worksheets"] = re.findall(
                r'<sheet[^>]+name="([^"]+)"',
                workbook_xml,
            )

        except (
            KeyError,
            zipfile.BadZipFile,
        ):
            pass

    # Legacy binary XLS
    elif content.startswith(
        b"\xD0\xCF\x11\xE0\xA1\xB1\x1A\xE1"
    ):
        result["format"] = "XLS Binary"

        try:
            workbook = pd.ExcelFile(path)

            result["worksheets"] = (
                workbook.sheet_names
            )

        except Exception:
            pass

    # PDF
    elif content.startswith(b"%PDF-"):
        result["format"] = "PDF"

    # Text-based formats
    else:
        text_content = content

        if text_content.startswith(
            b"\xEF\xBB\xBF"
        ):
            text_content = text_content[3:]

        try:
            text = text_content.decode(
                "utf-8"
            ).lstrip()

        except UnicodeDecodeError:
            result["format"] = (
                "Unknown binary format"
            )

        else:
            lowered = text.casefold()

            # SpreadsheetML XML
            if (
                "<ss:workbook" in lowered
                or (
                    text.startswith("<?xml")
                    and
                    "urn:schemas-microsoft-com:"
                    "office:spreadsheet"
                    in lowered
                )
            ):
                result["format"] = (
                    "SpreadsheetML XML"
                )

                result["worksheets"] = (
                    re.findall(
                        r'Worksheet\s+ss:Name="([^"]+)"',
                        text,
                    )
                )

            # Generic XML
            elif text.startswith("<?xml"):
                result["format"] = "XML"

            # HTML
            elif (
                lowered.startswith(
                    "<!doctype html"
                )
                or lowered.startswith(
                    "<html"
                )
            ):
                result["format"] = "HTML"

            else:
                result["format"] = (
                    "Unknown text format"
                )

    print(
        "Detected format:",
        result["format"],
    )

    print(
        "File extension:",
        result["extension"],
    )

    if result["worksheets"]:
        print(
            "Worksheets:",
            result["worksheets"],
        )

        print(
            "Worksheet count:",
            len(result["worksheets"]),
        )

    return result


benchmark_structure = inspect_file_structure(
    benchmark_file
)

Detected format: SpreadsheetML XML
File extension: .xls
Worksheets: ['Disclaimers', 'Holdings', 'Historical', 'Performance', 'Distributions']
Worksheet count: 5


# 5. DS-001 — Candidats

DS-001 est une préparation de DS-000 utilisée comme univers d’entrée pour les acquisitions nécessitant les instruments du benchmark.

In [10]:
security_candidates_file = get_dataset_path(
    "DS-001"
)

if not security_candidates_file.exists():
    raise FileNotFoundError(
        security_candidates_file
    )

security_candidates = pd.read_csv(
    security_candidates_file,
    dtype=str,
    keep_default_na=False,
)

if security_candidates.empty:
    status = "FAIL"
    message = "DS-001 est vide."

else:
    status = "PASS"
    message = (
        f"DS-001 contient "
        f"**{len(security_candidates):,} instruments**."
    )

display(
    security_candidates.head()
)

display(
    Markdown(
        f"{message}\n\n"
        f"**Résultat : {status}**"
    )
)

,Ticker,Name,Location,Exchange,Currency,Asset Class
0,MRVL,MARVELL TECHNOLOGY,United States,NASDAQ,USD,Equity
1,NVDA,NVIDIA,United States,NASDAQ,USD,Equity
2,6669,WIWYNN CORPORATION,Taiwan,Taiwan Stock Exchange,USD,Equity
3,TSN,TYSON FOODS INC CLASS A,United States,NYSE,USD,Equity
4,BMY,BRISTOL MYERS SQUIBB,United States,NYSE,USD,Equity


DS-001 contient **120 instruments**.

**Résultat : PASS**

In [11]:
exchange_summary = (
    security_candidates["Exchange"]
    .value_counts()
    .rename_axis("Exchange")
    .reset_index(name="securities")
)

print(
    "Distinct exchanges:",
    security_candidates["Exchange"].nunique(),
)

display(exchange_summary)

Distinct exchanges: 29


,Exchange,securities
0,Hong Kong Exchanges And Clearing Ltd,21
1,NASDAQ,14
2,Tokyo Stock Exchange,10
3,NYSE,8
4,Shenzhen Stock Exchange,7
5,Shanghai Stock Exchange,7
6,National Stock Exchange Of India,6
7,Nyse Euronext - Euronext Paris,5
8,Omx Nordic Exchange Copenhagen A/S,4
9,London Stock Exchange,4


# 6. DS-002 — Référentiel

On vérifie que le référentiel reprend chaque candidat, avec un symbole Yahoo et une devise de cotation.


In [12]:
security_master_file = get_dataset_path(
    "DS-002"
)

if not security_master_file.exists():
    raise FileNotFoundError(
        security_master_file
    )

security_master = pd.read_csv(
    security_master_file,
    dtype=str,
    keep_default_na=False,
)

if security_master.empty:
    status = "FAIL"
    message = "DS-002 est vide."

else:
    status = "PASS"
    message = (
        f"DS-002 contient "
        f"**{len(security_master):,} instruments**."
    )

display(
    security_master.head()
)

display(
    Markdown(
        f"{message}\n\n"
        f"**Résultat : {status}**"
    )
)

,Ticker,Name,Location,Exchange,Currency,listing_currency,Asset Class,yahoo_symbol
0,MRVL,MARVELL TECHNOLOGY,United States,NASDAQ,USD,USD,Equity,MRVL
1,NVDA,NVIDIA,United States,NASDAQ,USD,USD,Equity,NVDA
2,6669,WIWYNN CORPORATION,Taiwan,Taiwan Stock Exchange,USD,TWD,Equity,6669.TW
3,TSN,TYSON FOODS INC CLASS A,United States,NYSE,USD,USD,Equity,TSN
4,BMY,BRISTOL MYERS SQUIBB,United States,NYSE,USD,USD,Equity,BMY


DS-002 contient **120 instruments**.

**Résultat : PASS**

In [13]:
required = ["yahoo_symbol", "listing_currency"]
missing = [column for column in required if column not in security_master.columns]
if missing:
    status = "FAIL"
    message = "Colonnes absentes : " + ", ".join(missing)
else:
    empty_symbol = int(security_master["yahoo_symbol"].str.strip().eq("").sum())
    empty_currency = int(security_master["listing_currency"].str.strip().eq("").sum())
    duplicate_symbol = int(security_master["yahoo_symbol"].str.strip().duplicated().sum())
    if empty_symbol or empty_currency or duplicate_symbol:
        status = "FAIL"
        message = (
            f"Symboles vides : {empty_symbol}. "
            f"Devises vides : {empty_currency}. "
            f"Symboles répétés : {duplicate_symbol}."
        )
    else:
        status = "PASS"
        message = (
            f"Les {len(security_master)} lignes ont un symbole Yahoo "
            "et une devise de cotation, sans répétition."
        )
display(Markdown(f"{message}\n\n**Résultat : {status}**"))
display(
    security_master[
        ["Ticker", "Exchange", "listing_currency", "yahoo_symbol"]
    ].head(20)
)


Les 120 lignes ont un symbole Yahoo et une devise de cotation, sans répétition.

**Résultat : PASS**

,Ticker,Exchange,listing_currency,yahoo_symbol
0,MRVL,NASDAQ,USD,MRVL
1,NVDA,NASDAQ,USD,NVDA
2,6669,Taiwan Stock Exchange,TWD,6669.TW
3,TSN,NYSE,USD,TSN
4,BMY,NYSE,USD,BMY
5,VWS,Omx Nordic Exchange Copenhagen A/S,DKK,VWS.CO
6,FSLR,NASDAQ,USD,FSLR
7,9020,Tokyo Stock Exchange,JPY,9020.T
8,NOVN,SIX Swiss Exchange,CHF,NOVN.SW
9,1216,Taiwan Stock Exchange,TWD,1216.TW


# 7. DS-003 / SRC-002 — Cours Yahoo Finance

Cette section valide l'acquisition des données historiques de marché depuis Yahoo Finance, source principale du dataset `DS-003`.

La validation porte sur quatre dimensions :

1. **présence et exploitabilité de DS-003** ;
2. **couverture d'identité Yahoo** sur les 120 instruments configurés ;
3. **respect de la politique de résolution** (`DIRECT_VALIDATED`, `SEARCH_RESOLVED`, ambiguïtés) ;
4. **disponibilité et traçabilité des historiques effectivement acquis**.

La couverture n'est pas définie comme la capacité à trouver un symbole Yahoo pour chaque instrument.

Seuls les instruments `DIRECT_VALIDATED` sont automatiquement éligibles à l'acquisition. Les `SEARCH_RESOLVED` constituent des propositions de résolution soumises à revue et ne sont pas comptés comme données Yahoo acquises.

L'objectif est donc de mesurer la **couverture fiable** de Yahoo Finance, et non de forcer une couverture de 100 %.


In [14]:
market_prices_file = get_dataset_path(
    "DS-003"
)

if not market_prices_file.exists():
    raise FileNotFoundError(
        market_prices_file
    )

market_prices = pd.read_csv(
    market_prices_file,
    dtype=str,
    keep_default_na=False,
)

if market_prices.empty:
    status = "FAIL"
    message = "DS-003 est vide."
else:
    status = "PASS"
    message = (
        f"DS-003 contient "
        f"**{len(market_prices):,} observations de marché**."
    )

display(
    market_prices.head()
)

display(
    Markdown(
        f"{message}\n\n"
        f"**Résultat : {status}**"
    )
)

,input_position,input_index,configured_symbol,resolved_symbol,symbol_resolution_status,symbol,timestamp,open,high,low,close,adj_close,volume,dividends,stock_splits
0,41,41,688472.SS,688472.SS,DIRECT_VALIDATED,688472.SS,2023-06-09,12.899999618530273,13.600000381469727,11.960000038146973,13.050000190734863,12.616860389709473,283169140.0,0.0,0.0
1,41,41,688472.SS,688472.SS,DIRECT_VALIDATED,688472.SS,2023-06-12,12.979999542236328,15.130000114440918,12.8100004196167,14.670000076293945,14.183090209960938,208777787.0,0.0,0.0
2,41,41,688472.SS,688472.SS,DIRECT_VALIDATED,688472.SS,2023-06-13,14.350000381469727,15.670000076293945,14.350000381469727,14.800000190734863,14.308775901794434,133164505.0,0.0,0.0
3,41,41,688472.SS,688472.SS,DIRECT_VALIDATED,688472.SS,2023-06-14,14.670000076293945,15.5600004196167,14.640000343322754,15.1899995803833,14.685831069946289,86679311.0,0.0,0.0
4,41,41,688472.SS,688472.SS,DIRECT_VALIDATED,688472.SS,2023-06-15,15.100000381469727,16.850000381469727,14.699999809265137,16.399999618530273,15.855669021606445,116127819.0,0.0,0.0


DS-003 contient **649,659 observations de marché**.

**Résultat : PASS**

In [15]:
from src.acquisition.connectors.yfinance import DIRECT_VALIDATED_STATUS

coverage = (
    market_prices.loc[
        market_prices["symbol_resolution_status"].str.strip().eq(DIRECT_VALIDATED_STATUS),
        ["configured_symbol", "symbol_resolution_status"],
    ]
    .drop_duplicates()
    .reset_index(drop=True)
)
configured_count = int(
    security_master["yahoo_symbol"].astype(str).str.strip().ne("").sum()
)
acquired_count = int(
    coverage["configured_symbol"].astype(str).str.strip().nunique()
)
share = acquired_count / configured_count if configured_count else 0
status = "PASS" if acquired_count else "FAIL"
display(market_prices["symbol_resolution_status"].value_counts(dropna=False))
display(
    Markdown(
        f"**{acquired_count}** symboles directement validés "
        f"sur **{configured_count}** ({share:.2%}).\n\n"
        f"**Résultat : {status}**"
    )
)


symbol_resolution_status
DIRECT_VALIDATED    649659
Name: count, dtype: int64

**120** symboles directement validés sur **120** (100.00%).

**Résultat : PASS**

In [16]:
configured_symbols = set(
    security_master["yahoo_symbol"]
    .astype(str)
    .str.strip()
)

acquired_symbols = set(
    coverage["configured_symbol"]
    .astype(str)
    .str.strip()
)

not_acquired = (
    security_master.loc[
        security_master["yahoo_symbol"].isin(
            configured_symbols - acquired_symbols
        ),
        [
            "Ticker",
            "Name",
            "Exchange",
            "yahoo_symbol",
        ],
    ]
    .sort_values("yahoo_symbol")
    .reset_index(drop=True)
)

display(not_acquired)

status = (
    "PASS"
    if len(not_acquired)
    == configured_count - acquired_count
    else "FAIL"
)

display(
    Markdown(
        f"**{len(not_acquired)} instrument(s)** du référentiel "
        f"sont absents de DS-003.\n\n"
        f"Le connecteur ne télécharge que les symboles "
        f"`DIRECT_VALIDATED`.\n\n"
        f"**Résultat : {status}**"
    )
)

,Ticker,Name,Exchange,yahoo_symbol


**0 instrument(s)** du référentiel sont absents de DS-003.

Le connecteur ne télécharge que les symboles `DIRECT_VALIDATED`.

**Résultat : PASS**

In [17]:
history_coverage = (
    market_prices
    .groupby(
        "configured_symbol",
        as_index=False,
    )
    .agg(
        first_date=("timestamp", "min"),
        last_date=("timestamp", "max"),
        observations=("timestamp", "count"),
    )
)

missing_history = history_coverage.loc[
    history_coverage["observations"].eq(0)
]

status = (
    "PASS"
    if len(history_coverage) == acquired_count
    and missing_history.empty
    else "FAIL"
)

display(
    Markdown(
        f"Les **{len(history_coverage):,} instruments acquis** disposent "
        f"tous d'un historique de marché exploitable.\n\n"
        f"Période globale observée : "
        f"**{history_coverage['first_date'].min()}** à "
        f"**{history_coverage['last_date'].max()}**.\n\n"
        f"**Résultat : {status}**"
    )
)

display(
    history_coverage
    .sort_values("observations")
    .head(10)
    .reset_index(drop=True)
)

Les **120 instruments acquis** disposent tous d'un historique de marché exploitable.

Période globale observée : **1972-06-01** à **2026-11-02**.

**Résultat : PASS**

,configured_symbol,first_date,last_date,observations
0,3750.HK,2025-05-20,2026-11-02,361
1,688472.SS,2023-06-09,2026-11-02,822
2,301358.SZ,2023-02-09,2026-11-02,904
3,URW.PA,2023-04-14,2026-11-02,908
4,9863.HK,2022-09-29,2026-11-02,1004
5,9866.HK,2022-03-10,2026-11-02,1141
6,373220.KS,2022-01-27,2026-11-02,1161
7,2082.SR,2021-12-19,2026-11-02,1214
8,688187.SS,2021-09-07,2026-11-02,1245
9,RIVN,2021-11-10,2026-10-30,1248


# 8. DS-004 / SRC-003 — Cours de change

Cette section valide l'acquisition des cours de change CHF depuis le portail de données de la BNS pour le dataset `DS-004`.

Les devises demandées viennent de `listing_currency` dans DS-002. `snb_currency_codes` traduit chaque code ISO vers le code du cube BNS. La devise de reporting n'est pas demandée. La couverture de ce jeu est le nombre de séries reçues rapporté au nombre de séries demandées. Le cube `devkum` ne publie pas `ILS`, `INR`, `SAR` ni `TWD`. Ces devises sont absentes de `snb_currency_codes` et ne sont pas envoyées. La cellule les affiche à part, avec les titres qui les portent.

La sortie sépare deux unités. Une **série** est un cours de change (une devise). Un **titre** est une ligne du master. Plusieurs titres partagent la même série. Ces deux nombres ne s'additionnent pas.

Les cours restent dans l'unité publiée : `USD1` signifie 1 dollar, `JPY100` signifie 100 yens. La division par 100 n'est pas faite à l'acquisition.


In [18]:
fx_file = get_dataset_path("DS-004")

if not fx_file.exists():
    raise FileNotFoundError(fx_file)

fx_rates = pd.read_csv(
    fx_file,
    dtype=str,
    keep_default_na=False,
)

fx_row = datasets_registry.loc[
    datasets_registry["Dataset ID"].eq("DS-004")
]

if len(fx_row) != 1:
    raise ValueError(
        "Expected exactly one DS-004 registry row."
    )

raw_parameters = fx_row.iloc[0]["Parameters"]

fx_parameters = (
    json.loads(raw_parameters)
    if isinstance(raw_parameters, str)
    else dict(raw_parameters)
)

configured_cube = str(
    fx_parameters["cube_id"]
).strip()

security_master = pd.read_csv(
    get_dataset_path("DS-002"),
    dtype=str,
    keep_default_na=False,
)

currency_column = str(
    fx_parameters["currency_column"]
).strip()

reporting_currency = str(
    fx_parameters["reporting_currency"]
).strip()

configured_frequency_code = str(
    fx_parameters["frequency_code"]
).strip()

code_map = {
    str(currency).strip(): str(code).strip()
    for currency, code in fx_parameters["snb_currency_codes"].items()
}

listing_currencies = sorted(
    set(
        security_master[currency_column]
        .astype(str)
        .str.strip()
    )
)

unmapped_currencies = [
    currency
    for currency in listing_currencies
    if currency != reporting_currency
    and currency not in code_map
]

configured_currency_codes = {
    code_map[currency]
    for currency in listing_currencies
    if currency != reporting_currency
    and currency in code_map
}

fx_gap = (
    security_master.loc[
        security_master[currency_column].isin(
            unmapped_currencies
        ),
        ["Ticker", "Name", "Exchange", currency_column],
    ]
    .sort_values([currency_column, "Ticker"])
    .reset_index(drop=True)
)

def published_dimension_codes(series_key):
    match = re.fullmatch(
        r"[^{]+\{([^,]+),([^}]+)\}",
        str(series_key).strip(),
    )
    if match is None:
        raise ValueError(
            f"Unexpected SNB series key: {series_key}"
        )
    return match.group(1), match.group(2)

parsed_keys = fx_rates["series_key"].map(
    published_dimension_codes
)

observed_frequency_codes = {
    item[0] for item in parsed_keys
}

observed_currency_codes = {
    item[1] for item in parsed_keys
}

observed_series = set(
    fx_rates["series_key"].astype(str).str.strip()
)

duplicate_positions = fx_rates.duplicated(
    subset=["series_key", "value_position"],
    keep=False,
)

duplicate_dates = fx_rates.duplicated(
    subset=["series_key", "date"],
    keep=False,
)

missing_source_values = (
    fx_rates["value"].astype(str).str.strip().eq("")
)

numeric_values = pd.to_numeric(
    fx_rates["value"],
    errors="coerce",
)

series_summary = (
    fx_rates
    .assign(missing_source_value=missing_source_values)
    .groupby("series_key", as_index=False)
    .agg(
        first_date=("date", "min"),
        last_date=("date", "max"),
        observations=("date", "count"),
        missing_values=("missing_source_value", "sum"),
    )
    .sort_values("series_key")
    .reset_index(drop=True)
)

published_units = set(
    fx_rates["unit"].astype(str).str.strip()
)

published_labels = set(
    fx_rates["header_item_1"].astype(str).str.strip()
)

published_frequencies = set(
    fx_rates["frequency"].astype(str).str.strip()
)

contract_ok = (
    set(fx_rates["cube_id"].astype(str).str.strip())
    == {configured_cube}
    and observed_frequency_codes == {configured_frequency_code}
    and observed_currency_codes == configured_currency_codes
    and len(observed_series) == len(configured_currency_codes)
    and len(published_units) == 1
    and "" not in published_units
    and len(published_labels) == 1
    and "" not in published_labels
    and len(published_frequencies) == 1
    and "" not in published_frequencies
    and bool(numeric_values.notna().all())
)

status = (
    "PASS"
    if (
        not fx_rates.empty
        and not duplicate_positions.any()
        and not duplicate_dates.any()
        and contract_ok
    )
    else "FAIL"
)

code_to_currency = {
    code: currency
    for currency, code in code_map.items()
}

series_summary.insert(
    0,
    "listing_currency",
    series_summary["series_key"].map(
        lambda series_key: code_to_currency[
            published_dimension_codes(series_key)[1]
        ]
    ),
)

title_counts = (
    security_master[currency_column]
    .astype(str)
    .str.strip()
    .value_counts()
)

series_summary.insert(
    1,
    "titles",
    series_summary["listing_currency"].map(title_counts).astype(int),
)

mapped_currencies = [
    currency
    for currency in listing_currencies
    if currency != reporting_currency and currency in code_map
]

reporting_titles = int(
    security_master[currency_column]
    .astype(str)
    .str.strip()
    .eq(reporting_currency)
    .sum()
)

mapped_titles = int(
    security_master[currency_column]
    .astype(str)
    .str.strip()
    .isin(mapped_currencies)
    .sum()
)

coverage = pd.DataFrame(
    [
        {
            "groupe": "Série BNS publiée",
            "devises": len(mapped_currencies),
            "titres": mapped_titles,
            "note": (
                "Une série par devise. "
                "La colonne titres compte les lignes du master."
            ),
        },
        {
            "groupe": "Déjà en devise de reporting",
            "devises": int(reporting_currency in listing_currencies),
            "titres": reporting_titles,
            "note": (
                f"{reporting_currency} n'a pas de cours "
                f"{reporting_currency}/{reporting_currency}."
            ),
        },
        {
            "groupe": "Devise absente du cube BNS",
            "devises": len(unmapped_currencies),
            "titres": len(fx_gap),
            "note": (
                "Écart de couverture affiché ensuite. "
                "Ce n'est pas un échec du téléchargement."
            ),
        },
        {
            "groupe": "Total master DS-002",
            "devises": len(listing_currencies),
            "titres": len(security_master),
            "note": (
                "Somme des trois lignes. "
                "Devises et titres ne s'additionnent pas entre eux."
            ),
        },
    ]
)

missing_codes = ", ".join(unmapped_currencies) or "aucune"

display(
    Markdown(
        "**Comment lire ce résultat.** "
        "Une série est un cours de change. "
        "Un titre est une ligne du master. "
        "Plusieurs titres partagent la même série, "
        "donc le nombre de séries et le nombre de titres "
        "ne se combinent pas."
    )
)
display(coverage)

display(
    Markdown(
        "**Séries publiées.** "
        "Une ligne = une devise demandée à la BNS. "
        "`titles` indique combien de titres du master utilisent cette devise."
    )
)
display(series_summary)

display(
    Markdown(
        "**Titres sans cours BNS.** "
        "Une ligne = un titre. "
        f"Ces {len(fx_gap)} titres portent {len(unmapped_currencies)} devises "
        f"({missing_codes}), pas {len(fx_gap)} séries manquantes."
    )
)
display(fx_gap)

listing = security_master[currency_column].astype(str).str.strip()
received_currencies = {
    currency
    for currency, code in code_map.items()
    if code in observed_currency_codes
}
covered_titles = int(
    (
        listing.eq(reporting_currency)
        | listing.isin(received_currencies)
    ).sum()
)
title_count = len(security_master)
title_coverage = 100 * covered_titles / title_count

display(
    Markdown(
        f"DS-004 contient **{len(fx_rates):,} observations** "
        f"sur **{len(observed_series)} séries**, "
        f"soit les **{len(configured_currency_codes)} devises** "
        f"du master présentes dans `snb_currency_codes`.\n\n"
        f"**Cube :** {configured_cube}, code {configured_frequency_code}\n\n"
        f"**Période :** {fx_rates['date'].min()} → {fx_rates['date'].max()}\n\n"
        f"**Unité publiée :** {next(iter(published_units))}\n\n"
        f"**Valeurs vides :** {int(missing_source_values.sum()):,}\n\n"
        f"**Doublons techniques :** {int(duplicate_positions.sum()):,}\n\n"
        f"**Doublons série/date :** {int(duplicate_dates.sum()):,}\n\n"
        f"**Couverture des séries demandées :** "
        f"{len(observed_series)}/{len(configured_currency_codes)}.\n\n"
        f"**Pourcentage de couverture des {title_count} titres :** "
        f"{format(title_coverage, '.2f').replace('.', ',')} % "
        f"({covered_titles} titres).\n\n"
        f"**Résultat : {status}** — "
        "les séries demandées sont toutes présentes. "
        "Les devises absentes du cube restent dans le tableau du haut."
    )
)


**Comment lire ce résultat.** Une série est un cours de change. Un titre est une ligne du master. Plusieurs titres partagent la même série, donc le nombre de séries et le nombre de titres ne se combinent pas.

,groupe,devises,titres,note
0,Série BNS publiée,15,106,Une série par devise. La colonne titres compte...
1,Déjà en devise de reporting,1,3,CHF n'a pas de cours CHF/CHF.
2,Devise absente du cube BNS,4,11,Écart de couverture affiché ensuite. Ce n'est ...
3,Total master DS-002,20,120,Somme des trois lignes. Devises et titres ne s...


**Séries publiées.** Une ligne = une devise demandée à la BNS. `titles` indique combien de titres du master utilisent cette devise.

,listing_currency,titles,series_key,first_date,last_date,observations,missing_values
0,AUD,3,"EPB@SNB.devkum{M1,AUD1}",1992-01,2026-09,417,0
1,CAD,1,"EPB@SNB.devkum{M1,CAD1}",1970-01,2026-09,680,0
2,CNY,14,"EPB@SNB.devkum{M1,CNY100}",1998-12,2026-09,334,0
3,DKK,4,"EPB@SNB.devkum{M1,DKK100}",1970-01,2026-09,680,0
4,EUR,13,"EPB@SNB.devkum{M1,EUR1}",1999-01,2026-09,333,0
5,GBP,4,"EPB@SNB.devkum{M1,GBP1}",1970-01,2026-09,680,0
6,HKD,21,"EPB@SNB.devkum{M1,HKD100}",1999-01,2026-09,333,0
7,JPY,10,"EPB@SNB.devkum{M1,JPY100}",1972-01,2026-09,656,0
8,KRW,3,"EPB@SNB.devkum{M1,KRW100}",1999-01,2026-09,333,0
9,MXN,3,"EPB@SNB.devkum{M1,MXN100}",1999-01,2026-09,333,0


**Titres sans cours BNS.** Une ligne = un titre. Ces 11 titres portent 4 devises (ILS, INR, SAR, TWD), pas 11 séries manquantes.

,Ticker,Name,Exchange,listing_currency
0,ENLT,ENLIGHT RENEWABLE ENERGY,Tel Aviv Stock Exchange,ILS
1,COLPAL,COLGATE PALMOLIVE INDIA LTD,National Stock Exchange Of India,INR
2,HINDUNILVR,HINDUSTAN UNILEVER LTD,National Stock Exchange Of India,INR
3,LODHA,LODHA DEVELOPERS LTD,National Stock Exchange Of India,INR
4,LUPIN,LUPIN,National Stock Exchange Of India,INR
5,NESTLEIND,NESTLE INDIA LTD,National Stock Exchange Of India,INR
6,SUZLON,SUZLON ENERGY,National Stock Exchange Of India,INR
7,2082,ACWA POWER CO,Saudi Stock Exchange,SAR
8,1216,UNI-PRESIDENT ENTERPRISES,Taiwan Stock Exchange,TWD
9,6446,PHARMAESSENTIA,Taiwan Stock Exchange,TWD


DS-004 contient **7,563 observations** sur **15 séries**, soit les **15 devises** du master présentes dans `snb_currency_codes`.

**Cube :** devkum, code M1

**Période :** 1970-01 → 2026-09

**Unité publiée :** Rates at 11 am. in CHF

**Valeurs vides :** 0

**Doublons techniques :** 0

**Doublons série/date :** 0

**Couverture des séries demandées :** 15/15.

**Pourcentage de couverture des 120 titres :** 90,83 % (109 titres).

**Résultat : PASS** — les séries demandées sont toutes présentes. Les devises absentes du cube restent dans le tableau du haut.

# 9. DS-005 / SRC-003 – SARON

Cette section valide le taux SARON publié par la Banque nationale suisse. Le jeu est une API mensuelle, connecteur `snb`, cube `zimoma`, sans jeu amont.

La valeur reste le taux publié, en pour cent. Le taux mensuel sans risque n'est pas calculé ici.

DS-004 a retenu les titres qui ont une série BNS ou qui sont déjà en francs. Les titres sans série BNS n'entrent pas dans ce bilan. DS-005 mesure, sur les titres retenus, combien ont un cours en francs à chaque fin de mois de la fenêtre SARON.


In [19]:
saron_file = get_dataset_path("DS-005")

if not saron_file.exists():
    raise FileNotFoundError(saron_file)

saron = pd.read_csv(
    saron_file,
    dtype=str,
    keep_default_na=False,
)

saron_row = datasets_registry.loc[
    datasets_registry["Dataset ID"].eq("DS-005")
]

if len(saron_row) != 1:
    raise ValueError("Expected exactly one DS-005 registry row.")

registry_row = saron_row.iloc[0]

if str(registry_row["Production Method"]).strip() != "API":
    raise ValueError("DS-005 must be an API dataset.")

if str(registry_row["Connector"]).strip() != "snb":
    raise ValueError("DS-005 must use the snb connector.")

if str(registry_row["Input Dataset IDs"]).strip() not in {"[]", ""}:
    raise ValueError("DS-005 has no upstream dataset.")

values = pd.to_numeric(saron["value"], errors="coerce")
dates = pd.to_datetime(saron["date"], errors="coerce")

status = (
    "PASS"
    if (
        not saron.empty
        and saron["cube_id"].eq("zimoma").all()
        and saron["series_key"].str.contains("SARON").all()
        and values.notna().all()
        and dates.notna().all()
    )
    else "FAIL"
)

display(
    Markdown(
        f"DS-005 contient **{len(saron):,} observations** "
        f"du cube `zimoma`, série SARON.\n\n"
        f"Période : **{saron['date'].min()}** à **{saron['date'].max()}**.\n\n"
        f"Unité publiée : **{saron['unit'].iloc[0]}**.\n\n"
        f"**Résultat : {status}**"
    )
)

from src.config import ESTIMATION_MONTHS, MARKET_HISTORY_YEARS, MONTHS_PER_YEAR
from src.preparation.quality_treatment import (
    CHF_UNIVERSE,
    MARKET_PRICES_CHF,
    SARON_TREATED,
)

treated_saron = pd.read_csv(SARON_TREATED, dtype=str, keep_default_na=False)
treated_months = pd.PeriodIndex(
    treated_saron["date"].astype(str).str.slice(0, 7),
    freq="M",
)
window_end = treated_months.max()
scored_start = window_end - (MARKET_HISTORY_YEARS * MONTHS_PER_YEAR - 1)
history_start = scored_start - ESTIMATION_MONTHS
price_months = pd.period_range(history_start - 1, window_end, freq="M")
return_months = pd.period_range(history_start, window_end, freq="M")
saron_gap = [month for month in return_months if month not in set(treated_months)]

universe = pd.read_csv(CHF_UNIVERSE, dtype=str, keep_default_na=False)
prices = pd.read_csv(
    MARKET_PRICES_CHF,
    usecols=["symbol", "timestamp", "close_chf"],
)
prices["timestamp"] = pd.to_datetime(prices["timestamp"])
prices["month"] = prices["timestamp"].dt.to_period("M")
prices["close_chf"] = pd.to_numeric(prices["close_chf"], errors="coerce")
prices = prices.sort_values(["symbol", "timestamp"])
levels = prices.groupby(["symbol", "month"], as_index=False).tail(1)
levels = levels.loc[
    levels["month"].isin(set(price_months)) & levels["close_chf"].gt(0)
]
available = levels.groupby("symbol")["month"].nunique()
required_months = len(price_months)
universe["available_price_months"] = (
    universe["yahoo_symbol"].map(available).fillna(0).astype(int)
)
convertible = universe["chf_status"].eq("included")
complete = convertible & universe["available_price_months"].ge(required_months)
short = convertible & ~complete
master_count = len(universe)
convertible_count = int(convertible.sum())
complete_count = int(complete.sum())
short_count = int(short.sum())
without_series = master_count - convertible_count


def french_percent(part, whole):
    return format(100 * part / whole, ".2f").replace(".", ",")


coverage_balance = pd.DataFrame(
    [
        {
            "groupe": "Titres retenus par DS-004",
            "titres": convertible_count,
            "note": "Série BNS reçue, ou titre déjà en francs.",
        },
        {
            "groupe": "Historique complet",
            "titres": complete_count,
            "note": (
                f"Cours en francs strictement positif sur les "
                f"{required_months} fins de mois."
            ),
        },
        {
            "groupe": "Cotation plus courte",
            "titres": short_count,
            "note": "Hors de la fenêtre commune.",
        },
    ]
)

display(coverage_balance)
display(
    Markdown(
        f"**Fenêtre.** Cours exigés de **{price_months[0]}** à "
        f"**{price_months[-1]}** ({required_months} fins de mois). "
        f"SARON exigé de **{return_months[0]}** à **{return_months[-1]}** "
        f"({len(return_months)} mois). Mois de SARON manquants : "
        f"**{len(saron_gap)}**.\n\n"
        f"**Couverture SARON des {convertible_count} titres retenus :** "
        f"{french_percent(complete_count, convertible_count)} % "
        f"({complete_count} titres ont l'historique complet).\n\n"
        f"Les {without_series} titres sans série BNS restent hors de ce pourcentage."
    )
)


DS-005 contient **213 observations** du cube `zimoma`, série SARON.

Période : **2009-01** à **2026-09**.

Unité publiée : **In percent**.

**Résultat : PASS**

,groupe,titres,note
0,Titres retenus par DS-004,109,"Série BNS reçue, ou titre déjà en francs."
1,Historique complet,76,Cours en francs strictement positif sur les 18...
2,Cotation plus courte,33,Hors de la fenêtre commune.


**Fenêtre.** Cours exigés de **2011-09** à **2026-09** (181 fins de mois). SARON exigé de **2011-10** à **2026-09** (180 mois). Mois de SARON manquants : **0**.

**Couverture SARON des 109 titres retenus :** 69,72 % (76 titres ont l'historique complet).

Les 11 titres sans série BNS restent hors de ce pourcentage.

# 10. DS-006 / SRC-004 — Registre ODD

Cette section valide le registre d'évaluation ODD. SRC-004 n'a pas d'endpoint unique : les rapports annuels, les rapports de durabilité et les sites des sociétés ne passent pas par un connecteur.

La préparation part de DS-002 et ouvre une ligne par titre. Elle conserve les preuves déjà saisies et ajoute seulement les titres qui manquent. Le code n'écrit ni l'ODD, ni la contribution, ni le statut.

Le contrat de colonnes est celui de `src/preparation/odd_assessment.py`. Il comprend `Positive`. Il ne comprend pas `Data Quality`. La phase 4 vérifie le fichier. Elle ne juge pas si les phrases sont vraies.

L'identité du registre est `Ticker` + `Exchange`. FIGI et ISIN n'en sont pas des colonnes.

DS-005 a retenu les titres qui ont une série BNS, ou qui sont déjà en francs, et qui ont un cours à chaque fin de mois de la fenêtre SARON. Le bilan d'éligibilité porte sur ces titres. Le statut se saisit dans le CSV. Le programme ne l'attribue pas.


In [20]:
odd_file = get_dataset_path("DS-006")

if not odd_file.exists():
    raise FileNotFoundError(odd_file)

odd_assessment = pd.read_csv(
    odd_file,
    dtype=str,
    keep_default_na=False,
)

odd_row = datasets_registry.loc[
    datasets_registry["Dataset ID"].eq("DS-006")
]

if len(odd_row) != 1:
    raise ValueError(
        "Expected exactly one DS-006 registry row."
    )

if str(odd_row.iloc[0]["Production Method"]).strip() != "Preparation":
    raise ValueError("DS-006 must be a Preparation dataset.")

if str(odd_row.iloc[0]["Connector"]).strip() not in {"", "nan", "None"}:
    raise ValueError("DS-006 must not declare a connector.")

security_master = pd.read_csv(
    get_dataset_path("DS-002"),
    dtype=str,
    keep_default_na=False,
)

evidence_columns = [
    "SDG",
    "Activity",
    "Positive Contribution",
    "Negative Contribution",
    "Evidence",
    "Source",
    "Controversy",
    "Status",
]

expected_columns = [
    "Ticker",
    "Exchange",
    "Company",
    *evidence_columns,
    "Positive",
]

master_identity = set(
    zip(
        security_master["Ticker"].str.strip(),
        security_master["Exchange"].str.strip(),
    )
)

register_identity = set(
    zip(
        odd_assessment["Ticker"].str.strip(),
        odd_assessment["Exchange"].str.strip(),
    )
)

filled_evidence = odd_assessment[evidence_columns].ne("").any(axis=1)

duplicate_assessment = odd_assessment.duplicated(
    subset=["Ticker", "Exchange", "SDG"],
    keep=False,
)

missing_columns = [
    column for column in expected_columns
    if column not in odd_assessment.columns
]
extra_columns = [
    column for column in odd_assessment.columns
    if column not in expected_columns
]
allowed_status = {
    "",
    "Eligible",
    "Not_Eligible",
    "Indeterminate",
    "Excluded",
}
unknown_status = sorted(
    set(odd_assessment["Status"].str.strip()) - allowed_status
)

contract_ok = (
    not missing_columns
    and not extra_columns
    and not unknown_status
    and register_identity == master_identity
    and not duplicate_assessment.any()
)

status = "PASS" if contract_ok and not odd_assessment.empty else "FAIL"

coverage = pd.DataFrame(
    [
        {
            "groupe": "Titres du master",
            "lignes": len(security_master),
            "note": "Une ligne de registre est ouverte pour chacun.",
        },
        {
            "groupe": "Preuves déjà saisies",
            "lignes": int(filled_evidence.sum()),
            "note": "Le code ne remplit pas ces colonnes.",
        },
        {
            "groupe": "Preuves encore vides",
            "lignes": int((~filled_evidence).sum()),
            "note": "À compléter depuis les rapports de SRC-004.",
        },
    ]
)

display(coverage)
display(
    Markdown(
        f"DS-006 contient **{len(odd_assessment):,} lignes**, "
        f"soit **{len(register_identity)} titres** du master.\n\n"
        f"**Preuves saisies :** {int(filled_evidence.sum()):,} lignes\n\n"
        f"**Statut d'éligibilité vide :** "
        f"{int(odd_assessment['Status'].str.strip().eq('').sum()):,} lignes\n\n"
        f"**Doublons titre / place / ODD :** "
        f"{int(duplicate_assessment.sum()):,}\n\n"
        f"**Colonnes absentes du contrat :** {missing_columns or 'aucune'}\n\n"
        f"**Colonnes hors contrat :** {extra_columns or 'aucune'}\n\n"
        f"**Statuts hors vocabulaire :** {unknown_status or 'aucun'}\n\n"
        f"**Résultat : {status}** — le contrat du fichier est contrôlé. "
        "Ce résultat ne juge pas si les phrases sont vraies."
    )
)

from src.config import ESTIMATION_MONTHS, MARKET_HISTORY_YEARS, MONTHS_PER_YEAR
from src.preparation.quality_treatment import (
    CHF_UNIVERSE,
    MARKET_PRICES_CHF,
    SARON_TREATED,
)

treated_saron = pd.read_csv(SARON_TREATED, dtype=str, keep_default_na=False)
treated_months = pd.PeriodIndex(
    treated_saron["date"].astype(str).str.slice(0, 7),
    freq="M",
)
window_end = treated_months.max()
scored_start = window_end - (MARKET_HISTORY_YEARS * MONTHS_PER_YEAR - 1)
history_start = scored_start - ESTIMATION_MONTHS
price_months = set(pd.period_range(history_start - 1, window_end, freq="M"))
required_months = len(price_months)
chf_universe = pd.read_csv(CHF_UNIVERSE, dtype=str, keep_default_na=False)
franc_prices = pd.read_csv(
    MARKET_PRICES_CHF,
    usecols=["symbol", "timestamp", "close_chf"],
)
franc_prices["timestamp"] = pd.to_datetime(franc_prices["timestamp"])
franc_prices["month"] = franc_prices["timestamp"].dt.to_period("M")
franc_prices["close_chf"] = pd.to_numeric(franc_prices["close_chf"], errors="coerce")
franc_prices = franc_prices.sort_values(["symbol", "timestamp"])
month_end = franc_prices.groupby(["symbol", "month"], as_index=False).tail(1)
month_end = month_end.loc[
    month_end["month"].isin(price_months) & month_end["close_chf"].gt(0)
]
available_months = month_end.groupby("symbol")["month"].nunique()
chf_universe["available_price_months"] = (
    chf_universe["yahoo_symbol"].map(available_months).fillna(0).astype(int)
)
retained_names = chf_universe.loc[
    chf_universe["chf_status"].eq("included")
    & chf_universe["available_price_months"].ge(required_months),
    ["ticker", "exchange"],
]
retained_register = odd_assessment.merge(
    retained_names,
    left_on=["Ticker", "Exchange"],
    right_on=["ticker", "exchange"],
    how="inner",
)
decided_status = {"Eligible", "Not_Eligible", "Excluded"}
decided = retained_register["Status"].str.strip().isin(decided_status)
retained_count = len(retained_register)
decided_count = int(decided.sum())
open_count = int(retained_register["Status"].str.strip().eq("Indeterminate").sum())


def french_percent(part, whole):
    if whole == 0:
        return "0,00"
    return format(100 * part / whole, ".2f").replace(".", ",")


eligibility = pd.DataFrame(
    [
        {
            "groupe": "Titres retenus par DS-005",
            "titres": retained_count,
            "note": "Série BNS ou francs, et historique complet.",
        },
        {
            "groupe": "Statut encore Indeterminate",
            "titres": open_count,
            "note": "Statut saisi. Cette valeur ne donne pas de poids.",
        },
        {
            "groupe": "Statut décidé",
            "titres": decided_count,
            "note": "Eligible, Not_Eligible ou Excluded.",
        },
    ]
)
display(eligibility)
display(
    Markdown(
        f"**Couverture ODD des {retained_count} titres retenus :** "
        f"{french_percent(decided_count, retained_count)} % "
        f"ont un statut décidé.\n\n"
        "Le statut se saisit dans `data/01_raw/odd/odd_assessment.csv`. "
        "Le programme ne l'attribue pas."
    )
)

,groupe,lignes,note
0,Titres du master,120,Une ligne de registre est ouverte pour chacun.
1,Preuves déjà saisies,120,Le code ne remplit pas ces colonnes.
2,Preuves encore vides,0,À compléter depuis les rapports de SRC-004.


DS-006 contient **120 lignes**, soit **120 titres** du master.

**Preuves saisies :** 120 lignes

**Statut d'éligibilité vide :** 0 lignes

**Doublons titre / place / ODD :** 0

**Colonnes absentes du contrat :** aucune

**Colonnes hors contrat :** aucune

**Statuts hors vocabulaire :** aucun

**Résultat : PASS** — le contrat du fichier est contrôlé. Ce résultat ne juge pas si les phrases sont vraies.

,groupe,titres,note
0,Titres retenus par DS-005,76,"Série BNS ou francs, et historique complet."
1,Statut encore Indeterminate,38,Statut saisi. Cette valeur ne donne pas de poids.
2,Statut décidé,38,"Eligible, Not_Eligible ou Excluded."


**Couverture ODD des 76 titres retenus :** 50,00 % ont un statut décidé.

Le statut se saisit dans `data/01_raw/odd/odd_assessment.csv`. Le programme ne l'attribue pas.

# 11. Journal et traçabilité

## Technical Log

Le Technical Log documente les étapes d’exécution, validations, warnings et erreurs techniques.


In [21]:
if TECHNICAL_LOG.exists():
    with open(TECHNICAL_LOG, "r", encoding="utf-8", errors="ignore") as handle:
        log_lines = handle.readlines()
    print("Technical log lines:", len(log_lines))
    print("".join(log_lines[-20:]))
else:
    print("Technical log not found:", TECHNICAL_LOG)

Technical log lines: 6306
2026-11-02 10:34:39,650 | INFO | Database schema initialization completed successfully
2026-11-02 10:34:39,659 | INFO | [AUDIT] RUN-20261102-093439-15E8D313 | Preparation | saron.csv | Success | Records=213
2026-11-02 10:34:39,659 | INFO | Quality treatment published 213 rows to C:\Users\MameBARBOSAMONTEIRO\projet01\data\02_processed\macro\saron.csv (RUN-20261102-093439-15E8D313)
2026-11-02 10:34:39,678 | INFO | Database schema initialization started
2026-11-02 10:34:39,681 | INFO | Database migration already applied: 1
2026-11-02 10:34:39,681 | INFO | Database migration already applied: 2
2026-11-02 10:34:40,144 | INFO | Database schema version: 2
2026-11-02 10:34:40,144 | INFO | Technical table available: acquisition_log
2026-11-02 10:34:40,144 | INFO | Technical table available: dataset_registry
2026-11-02 10:34:40,144 | INFO | Technical table available: quality_run
2026-11-02 10:34:40,144 | INFO | Technical table available: quality_control_result
2026-11-0

## Acquisition / Preparation Log

La validation du log métier doit s’appuyer sur la table ou le fichier réellement utilisé par `acquisition_logger.py`. Le workbook ne suppose plus l’ancien contrat Excel si le mécanisme courant a évolué.

In [22]:
if DATABASE_FILE.exists():
    with sqlite3.connect(DATABASE_FILE) as connection:
        tables = set(pd.read_sql("SELECT name FROM sqlite_master WHERE type='table'", connection)["name"])
        if "acquisition_log" in tables:
            acquisition_log = pd.read_sql(
                "SELECT * FROM acquisition_log ORDER BY execution_date DESC LIMIT 30",
                connection,
            )
            display(acquisition_log)
        else:
            print("acquisition_log table not found.")

,run_id,execution_date,process_type,output_file,storage_location,status,records_produced,notes,created_at
0,RUN-20261102-093439-15E8D313,2026-11-02T09:34:39+00:00,Preparation,saron.csv,C:\Users\MameBARBOSAMONTEIRO\projet01\data\02_...,Success,213.0,"{""empty_values"": 0, ""rate_currency"": ""CHF"", ""s...",2026-11-02T09:34:39+00:00
1,RUN-20261102-093439-D33B7494,2026-11-02T09:34:39+00:00,Preparation,chf_portfolio_universe.csv,C:\Users\MameBARBOSAMONTEIRO\projet01\data\03_...,Success,120.0,"{""excluded"": 11, ""included"": 109}",2026-11-02T09:34:39+00:00
2,RUN-20261102-093438-9FC73B89,2026-11-02T09:34:38+00:00,Preparation,market_price_exclusions.csv,C:\Users\MameBARBOSAMONTEIRO\projet01\data\02_...,Success,61843.0,"{""missing_close"": 1123, ""missing_monthly_fx"": ...",2026-11-02T09:34:38+00:00
3,RUN-20261102-093437-7DE1F837,2026-11-02T09:34:37+00:00,Preparation,market_prices_chf.csv,C:\Users\MameBARBOSAMONTEIRO\projet01\data\02_...,Success,587816.0,"{""excluded_rows"": 61843, ""kept_rows"": 587816, ...",2026-11-02T09:34:37+00:00
4,RUN-20261102-093419-36AEC8A3,2026-11-02T09:34:19+00:00,Preparation,quote_currency.csv,C:\Users\MameBARBOSAMONTEIRO\projet01\data\02_...,Success,120.0,"{""currencies_absent"": 0, ""currencies_present"":...",2026-11-02T09:34:19+00:00
5,RUN-20261102-093313-AAEB0A3F,2026-11-02T09:33:13+00:00,Preparation,saron.csv,C:\Users\MameBARBOSAMONTEIRO\projet01\data\02_...,Success,213.0,"{""empty_values"": 0, ""rate_currency"": ""CHF"", ""s...",2026-11-02T09:33:13+00:00
6,RUN-20261102-093313-E4AC8C94,2026-11-02T09:33:13+00:00,Preparation,chf_portfolio_universe.csv,C:\Users\MameBARBOSAMONTEIRO\projet01\data\03_...,Success,120.0,"{""excluded"": 11, ""included"": 109}",2026-11-02T09:33:13+00:00
7,RUN-20261102-093312-F0DD6F22,2026-11-02T09:33:12+00:00,Preparation,market_price_exclusions.csv,C:\Users\MameBARBOSAMONTEIRO\projet01\data\02_...,Success,71868.0,"{""missing_close"": 1123, ""missing_monthly_fx"": ...",2026-11-02T09:33:12+00:00
8,RUN-20261102-093311-BAD4FCE3,2026-11-02T09:33:11+00:00,Preparation,market_prices_chf.csv,C:\Users\MameBARBOSAMONTEIRO\projet01\data\02_...,Success,577791.0,"{""excluded_rows"": 71868, ""kept_rows"": 577791, ...",2026-11-02T09:33:11+00:00
9,RUN-20261102-093137-5BBF2007,2026-11-02T09:31:37+00:00,Preparation,chf_portfolio_universe.csv,C:\Users\MameBARBOSAMONTEIRO\projet01\data\03_...,Success,120.0,"{""excluded"": 11, ""included"": 109}",2026-11-02T09:31:37+00:00


# 12. Base — état courant

Le chargement SQLite est une étape distincte de l’acquisition. Ce contrôle inventorie les tables réellement disponibles sans exiger des tables historiques supprimées par le refactoring.


In [23]:
if DATABASE_FILE.exists():
    with sqlite3.connect(DATABASE_FILE) as connection:
        database_tables = pd.read_sql(
            "SELECT name FROM sqlite_master WHERE type='table' ORDER BY name",
            connection,
        )
    database_tables
else:
    print("Database not found.")

# 13. Couverture globale de la Phase 3

Ce tableau est mis à jour à mesure que chaque source est définitivement validée.


In [24]:
phase3_status = pd.DataFrame([
    {"Source": "SRC-001", "Jeu": "DS-000", "Domaine": "Feuille de positions", "Situation": "Validé"},
    {"Source": "SRC-001", "Jeu": "DS-001", "Domaine": "Candidats", "Situation": "Validé"},
    {"Source": "SRC-001", "Jeu": "DS-002", "Domaine": "Référentiel", "Situation": "Validé"},
    {"Source": "SRC-002", "Jeu": "DS-003", "Domaine": "Cours et volume", "Situation": "Validé"},
    {"Source": "SRC-003", "Jeu": "DS-004", "Domaine": "Change", "Situation": "Validé"},
    {"Source": "SRC-003", "Jeu": "DS-005", "Domaine": "SARON", "Situation": "Validé"},
    {"Source": "SRC-004", "Jeu": "DS-006", "Domaine": "Registre ODD", "Situation": "Ouvert"},
])
phase3_status


,Source,Jeu,Domaine,Situation
0,SRC-001,DS-000,Feuille de positions,Validé
1,SRC-001,DS-001,Candidats,Validé
2,SRC-001,DS-002,Référentiel,Validé
3,SRC-002,DS-003,Cours et volume,Validé
4,SRC-003,DS-004,Change,Validé
5,SRC-003,DS-005,SARON,Validé
6,SRC-004,DS-006,Registre ODD,Ouvert


# 14. Critères de clôture de la Phase 3

La phase 3 couvre les sources qui alimentent les phases suivantes.

1. Chaque source du registre a un jeu exploitable.
2. Chaque jeu automatisable a un connecteur ou un téléchargement.
3. Les fichiers reçus existent.
4. Les dépendances sont respectées.
5. Les exécutions sont tracées.
6. Les jeux chargés en base ont un contrat de chargement.
7. Le registre ODD et les exclusions ne sont pas remplis par un connecteur.


# Conclusion

Les acquisitions automatisées retenues sont iShares (`DS-000`), Yahoo Finance (`DS-003`) et la Banque nationale suisse (`DS-004`, `DS-005`).

La chaîne `DS-000 → DS-001 → DS-002 → DS-003` fournit les cours. Le change et le SARON viennent de la Banque nationale suisse. Le registre ODD (`DS-006`) est tenu à partir des rapports de société. Aucun connecteur ne lui attribue un statut.

L'orchestrateur `python -m src.pipeline.refresh` reprend chaque jeu actif selon sa fréquence dans le registre.
